# Tokenización y Representaciones Vectoriales en LLMs

## ¿Qué es la tokenización?

Antes de que un modelo de lenguaje como GPT o BERT pueda procesar texto, necesita convertir las palabras (o partes de palabras) en números. Este proceso se llama **tokenización**.

Un **token** es la unidad básica de texto que el modelo "ve". Puede ser:
- Una palabra completa: `"perro"` → token
- Un subpalabra: `"tokenización"` → `["token", "ización"]`
- Un carácter individual

## ¿Qué es un embedding (representación vectorial)?

Una vez tokenizado el texto, cada token se mapea a un **vector de números reales** en un espacio de alta dimensión. Este vector es el *embedding* del token.

**¿Por qué vectores?** Porque nos permiten representar el *significado semántico* de las palabras de forma matemática. Palabras con significados similares quedan **cerca** en este espacio vectorial.

```
"rey"   → [0.23, -0.81, 0.45, ..., 0.12]  (300 números)
"reina" → [0.21, -0.79, 0.48, ..., 0.15]  (300 números, ¡muy similar!)
"mesa"  → [-0.54, 0.33, -0.12, ..., 0.87] (300 números, muy diferente)
```

En este notebook usaremos **Word2Vec** (Google News), un modelo clásico entrenado sobre millones de artículos de noticias, para explorar estas representaciones.

## Paso 1 ? Dependencia

El modelo preentrenado ocupa aproximadamente 1.6 GB. Este notebook **no inicia su descarga autom?ticamente**. Cuando el modelo est? disponible localmente, instala `gensim` si hace falta con `python -m pip install gensim` y vuelve a ejecutar desde el inicio.


In [1]:
import importlib.util
GENSIM_AVAILABLE = importlib.util.find_spec("gensim") is not None
print("gensim instalado:", "s?" if GENSIM_AVAILABLE else "no")


gensim instalado: no


## Paso 2 — Cargar el modelo Word2Vec

Cargamos el modelo `word2vec-google-news-300`, entrenado por Google sobre ~100 mil millones de palabras de artículos de noticias.

- Contiene ~3 millones de palabras/frases en su vocabulario
- Cada token está representado por un vector de **300 dimensiones**

> **Nota:** La primera vez esto descarga ~1.6 GB. Después queda en caché local.

In [2]:
import os
from pathlib import Path

MODEL_NAME = "word2vec-google-news-300"
GENSIM_DATA_DIR = Path(os.environ.get("GENSIM_DATA_DIR", str(Path.home() / "gensim-data")))
MODEL_CACHE_DIR = GENSIM_DATA_DIR / MODEL_NAME
_cached_files = [
    path for path in MODEL_CACHE_DIR.rglob("*")
    if path.is_file() and path.suffix.lower() in {".gz", ".bin"}
    and path.stat().st_size >= 1_500_000_000
] if MODEL_CACHE_DIR.is_dir() else []

if not _cached_files:
    model = None
    print(f"No se encontr? el archivo completo de {MODEL_NAME} en la cach? {MODEL_CACHE_DIR}.")
    print("Carga omitida para evitar descargar autom?ticamente el modelo (~1.6 GB).")
elif not GENSIM_AVAILABLE:
    model = None
    print("El archivo est? en cach?, pero gensim no est? instalado; instala gensim y vuelve a ejecutar.")
else:
    # Carga el archivo local directamente: no usa api.load, que descarga si falta el recurso.
    from gensim.models import KeyedVectors
    MODEL_FILE = max(_cached_files, key=lambda path: path.stat().st_size)
    model = KeyedVectors.load_word2vec_format(str(MODEL_FILE), binary=True)
    print(f"Modelo cargado desde la cach? local: {MODEL_FILE}")


No se encontr? el archivo completo de word2vec-google-news-300 en la cach? C:\Users\juand\gensim-data\word2vec-google-news-300.
Carga omitida para evitar descargar autom?ticamente el modelo (~1.6 GB).


## Paso 3 — ¿Cuántas dimensiones tiene un embedding?

Verifiquemos la dimensión del vector de la palabra `"apple"`. Cada token en este modelo está representado como un punto en un espacio de **300 dimensiones**.

Aunque 300 dimensiones es imposible de visualizar directamente, podemos usar técnicas de reducción dimensional como **PCA** (Análisis de Componentes Principales) o **t-SNE** para proyectarlos en 2D o 3D y así visualizar las relaciones semánticas entre palabras.

In [3]:
if model is None:
    print("Actividad omitida: el modelo Word2Vec no est? disponible localmente.")
else:
    print("Dimensiones del embedding apple:", len(model['apple']))

Actividad omitida: el modelo Word2Vec no est? disponible localmente.


## Paso 4 — Aritmética vectorial: la magia de los embeddings

Una de las propiedades más fascinantes de los embeddings es que **las relaciones semánticas se preservan como operaciones vectoriales**.

El experimento clásico es:

$$\vec{king} - \vec{man} + \vec{woman} \approx \vec{queen}$$

¿Qué significa esto? El vector que va de `"man"` a `"king"` (la *dirección de realeza*) es aproximadamente el mismo que el que va de `"woman"` a `"queen"`. El modelo aprendió este concepto implícitamente al leer millones de textos — **nadie le enseñó explícitamente qué es un rey o una reina**.

### Visualización con PCA 3D

Como no podemos graficar 300 dimensiones, usamos **PCA** para reducir a 3D, buscando las tres direcciones que capturan la mayor varianza. Las líneas rojas y azules muestran la similitud coseno entre el vector resultado y `king`/`queen`.

In [4]:
if model is None:
    print("Actividad omitida: el modelo Word2Vec no est? disponible localmente.")
else:
    from sklearn.decomposition import PCA
    from sklearn.metrics.pairwise import cosine_similarity
    import matplotlib.pyplot as plt
    from mpl_toolkits.mplot3d import Axes3D
    import numpy as np


    v_king = model['king']
    v_man = model['man']
    v_woman = model['woman']
    v_queen = model['queen']
    v_result = v_king - v_man + v_woman


    words_to_plot = ['king', 'man', 'queen', 'woman', 'result']
    vectors_to_plot = [v_king, v_man, v_queen, v_woman, v_result]

    pca = PCA(n_components=3)
    reduced = pca.fit_transform(vectors_to_plot)
    coords = {word: reduced[i] for i, word in enumerate(words_to_plot)}

    def cos(a, b):
        return cosine_similarity([a], [b])[0, 0]

    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')

    for word, vec in coords.items():
        ax.scatter(vec[0], vec[1], vec[2])
        ax.text(vec[0], vec[1], vec[2], word)

    result = coords['result']
    king = coords['king']
    queen = coords['queen']

    # Result ↔ King
    ax.plot([result[0], king[0]], [result[1], king[1]], [result[2], king[2]], color='red')
    mid_rk = (result + king) / 2.0
    ax.text(mid_rk[0], mid_rk[1], mid_rk[2],
            f"sim={cos(v_result, v_king):.2f}", color='red')

    # King ↔ Queen
    ax.plot([king[0], queen[0]], [king[1], queen[1]], [king[2], queen[2]], color='red')
    mid_kq = (king + queen) / 2.0
    ax.text(mid_kq[0], mid_kq[1], mid_kq[2],
            f"sim={cos(v_king, v_queen):.2f}", color='red')

    # Result ↔ Queen
    ax.plot([result[0], queen[0]], [result[1], queen[1]], [result[2], queen[2]], color='blue')
    mid_rq = (result + queen) / 2.0
    ax.text(mid_rq[0], mid_rq[1], mid_rq[2],
            f"sim={cos(v_result, v_queen):.2f}", color='blue')

    ax.set_title("3D PCA — King - Man + Woman = Result")
    ax.set_xlabel("PCA 1"); ax.set_ylabel("PCA 2"); ax.set_zlabel("PCA 3")
    plt.show()

Actividad omitida: el modelo Word2Vec no est? disponible localmente.


## Paso 5 — ¿A qué palabras se parece el resultado?

Ahora buscamos los 10 tokens más similares al vector resultado de `king - man + woman` usando **similitud coseno**.

La similitud coseno mide el ángulo entre dos vectores en el espacio de embeddings:
- `sim = 1.0` → misma dirección (palabras idénticas o sinónimos perfectos)
- `sim ≈ 0.7` → fuertemente relacionadas
- `sim ≈ 0.0` → sin relación semántica

Si los embeddings capturaron bien la semántica, deberíamos ver `"queen"` entre los primeros resultados — y de hecho aparece en el segundo lugar.

In [5]:
if model is None:
    print("Actividad omitida: el modelo Word2Vec no est? disponible localmente.")
else:
    similar_words = model.most_similar(positive=[v_result], topn=10)

    print("Top 10 palabras mas similares a la operación 'king - man + woman':")
    for word, similarity in similar_words:
        print(f"{word}: {similarity:.4f}")

Actividad omitida: el modelo Word2Vec no est? disponible localmente.


### Vecinos más cercanos de `"queen"`

Para comparar, veamos directamente los 10 tokens más similares a `"queen"`. Esto nos dice qué conceptos el modelo asocia con ser reina. Observa que `king` aparece como vecino de `queen` y viceversa — los embeddings capturaron la relación bidireccional entre estos conceptos.

In [6]:
if model is None:
    print("Actividad omitida: el modelo Word2Vec no est? disponible localmente.")
else:
    similar_words = model.most_similar(positive=[v_queen], topn=10)

    print("Top 10 palabras mas similares a la operación 'queen':")
    for word, similarity in similar_words:
        print(f"{word}: {similarity:.4f}")

Actividad omitida: el modelo Word2Vec no est? disponible localmente.


### Vecinos más cercanos de `"king"`

Finalmente, los 10 tokens más similares a `"king"`. Compara los vecinos de `king` vs. los de `queen`: ¿son simétricos? ¿Hay sesgos del corpus de entrenamiento que puedas identificar?

In [7]:
if model is None:
    print("Actividad omitida: el modelo Word2Vec no est? disponible localmente.")
else:
    similar_words = model.most_similar(positive=[v_king], topn=10)

    print("Top 10 palabras mas similares a la operación 'king':")
    for word, similarity in similar_words:
        print(f"{word}: {similarity:.4f}")

Actividad omitida: el modelo Word2Vec no est? disponible localmente.


---
## Reto: Explora tu propio espacio semántico

Ahora es tu turno. Usando el modelo Word2Vec que ya cargamos, realiza las siguientes exploraciones:

### Parte 1 — Tu animal favorito
1. Elige tu animal favorito en inglés (e.g., `"dolphin"`, `"tiger"`, `"eagle"`)
2. Obtén su vector: `v_animal = model["nombre_del_animal"]`
3. Busca los 10 tokens más similares
4. ¿Los resultados tienen sentido semántico? ¿Encuentras tokens sorprendentes?

### Parte 2 — Operaciones vectoriales
Diseña operaciones análogas a `king - man + woman` con tu animal:
- `"dog" - "domestic" + "wild"` → ¿a qué llega?
- `"eagle" - "fly" + "swim"` → ¿qué predice el modelo?
- Inventa tu propia operación semántica y explica qué esperabas y qué obtuviste

### Parte 3 — Visualización
Toma tu animal, sus vecinos más cercanos y alguna operación vectorial, y grafícalos en 3D con PCA igual que hicimos arriba. ¿Forman clusters semánticos con sentido?

> **Tip:** Si una palabra no está en el vocabulario, el modelo lanzará un `KeyError`. Verifica con `"palabra" in model.key_to_index` antes de usarla.

In [8]:
# === RETO: exploraci?n lista para ejecutarse cuando el modelo est? en cach? ===

if model is None:
    print("Reto pendiente: primero instala gensim si hace falta y coloca word2vec-google-news-300 en su cach? local; no se descargar? aqu?.")
else:
    # ----- Parte 1: animal favorito -----
    mi_animal = "dolphin"
    if mi_animal not in model.key_to_index:
        raise KeyError(f"{mi_animal!r} no est? en el vocabulario; elige otra palabra en ingl?s.")
    v_animal = model[mi_animal]
    animal_neighbors = model.most_similar(positive=[v_animal], topn=10)
    print(f"Vector dimension of {mi_animal!r}: {len(v_animal)}")
    print(f"\nTop 10 most similar words to {mi_animal!r}:")
    for word, sim in animal_neighbors:
        print(f"  {word}: {sim:.4f}")

    # ----- Parte 2: ejemplos y operaci?n propia -----
    operations = [
        ("dog - domestic + wild", ["dog", "wild"], ["domestic"]),
        ("eagle - fly + swim", ["eagle", "swim"], ["fly"]),
        ("paris - france + italy", ["paris", "italy"], ["france"]),
    ]
    operation_vectors = {}
    operation_neighbors = {}
    for label, positive, negative in operations:
        missing = [word for word in positive + negative if word not in model.key_to_index]
        if missing:
            print(f"\nSe omite {label}; fuera de vocabulario: {missing}")
            continue
        vector = sum((model[word] for word in positive), start=np.zeros(model.vector_size))
        vector -= sum((model[word] for word in negative), start=np.zeros(model.vector_size))
        operation_vectors[label] = vector
        neighbors = model.most_similar(positive=[vector], topn=5)
        operation_neighbors[label] = neighbors
        print(f"\n{label} ?")
        for word, sim in neighbors:
            print(f"  {word}: {sim:.4f}")

    # ----- Parte 3: PCA 3D de dolphin, vecinos y una operaci?n -----
    from sklearn.decomposition import PCA
    import matplotlib.pyplot as plt

    plot_entries = [(mi_animal, v_animal)]
    for word, _ in animal_neighbors[:6]:
        if word in model.key_to_index:
            plot_entries.append((word, model[word]))
    for word in ["ocean", "sea", "swim", "mammal", "whale"]:
        if word in model.key_to_index:
            plot_entries.append((word, model[word]))
    chosen_operation = "dog - domestic + wild"
    if chosen_operation in operation_vectors:
        plot_entries.append((chosen_operation, operation_vectors[chosen_operation]))
        for word, _ in operation_neighbors[chosen_operation][:3]:
            plot_entries.append((word, model[word]))

    # Quita etiquetas repetidas antes de ajustar PCA.
    unique_entries = {}
    for label, vector in plot_entries:
        unique_entries.setdefault(label, vector)
    labels = list(unique_entries)
    vectors = np.vstack([unique_entries[label] for label in labels])
    if len(labels) < 3:
        raise ValueError("Se necesitan al menos tres vectores distintos para visualizar PCA en 3D.")
    reduced = PCA(n_components=3).fit_transform(vectors)

    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection="3d")
    for label, point in zip(labels, reduced):
        ax.scatter(*point)
        ax.text(*point, label)
    ax.set_title(f"3D PCA ? vecinos sem?nticos de {mi_animal!r}")
    ax.set_xlabel("PCA 1"); ax.set_ylabel("PCA 2"); ax.set_zlabel("PCA 3")
    plt.show()


Reto pendiente: primero instala gensim si hace falta y coloca word2vec-google-news-300 en su cach? local; no se descargar? aqu?.


## Reflexi?n del reto ? estado de esta ejecuci?n

El espacio sem?ntico no se ejecut? en este entorno: no est? disponible la cach? local de `word2vec-google-news-300` y la descarga de aproximadamente 1.6 GB se dej? desactivada. Por ello no afirmo vecinos, resultados de operaciones ni clusters PCA. Al cargar el modelo se podr?n completar esas observaciones con las salidas reales de las celdas anteriores y del reto.